In [2]:
# Make sure we have bitsandbytes installed
!pip list installed | grep bitsandbytes

bitsandbytes                          0.50.2


**D1 design**

We'll use:

* GPT-2 124M
* 4-bit NF4 quantization
* double quantization
* LoRA only on the MLP
    * mlp.c_fc
    * mlp.c_proj
* applied to all 12 Transformer blocks
* adult → child conversational pairs only
* response-only SFT
* 10k / 1k / 1k train/validation/test
* same r=8, alpha=16, dropout 0.05
* same LR 5e-4
* same batch size and 1 epoch




So, the progression is now:

**C1**:
FP32 base
MLP LoRA, h11, the last attention block

**C2**:
FP32 base
MLP LoRA, all blocks

**C3**:
FP32 base
Attention + MLP LoRA, all blocks

**D1**:
4-bit NF4 base
MLP LoRA, all blocks

**D2**:
4-bit NF4 base
Attention + MLP LoRA, all blocks

In [3]:
import torch

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

In [4]:
# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "openai-community/gpt2"
DATASET_NAME = "nshah-fbcs/childes-engUK-conversational-pairs"

OUTPUT_DIR = "./gpt2-childes-qlora-d1"

MAX_LENGTH = 1024

TRAIN_SIZE = 10_000
VAL_SIZE = 1_000
TEST_SIZE = 1_000

SEED = 42


In [5]:
# ============================================================
# LoRA configuration
# ============================================================

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05



In [6]:
# ============================================================
# Training configuration
# ============================================================

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
BATCH_SIZE = 4
NUM_EPOCHS = 1



In [7]:
# ============================================================
# 4-bit QLoRA configuration
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,

    # QLoRA's recommended 4-bit quantization type.
    bnb_4bit_quant_type="nf4",

    # Double quantization:
    # quantize the quantization constants themselves.
    bnb_4bit_use_double_quant=True,

    # Compute in float32 on this simple experiment.
    bnb_4bit_compute_dtype=torch.float32,
)


In [8]:
# ============================================================
# Load tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# GPT-2 has no dedicated padding token.
tokenizer.pad_token = tokenizer.eos_token



In [9]:
# ============================================================
# Load dataset
# ============================================================

dataset = load_dataset(DATASET_NAME)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['speaker_a', 'utterance_a', 'speaker_b', 'utterance_b'],
        num_rows: 250757
    })
    validation: Dataset({
        features: ['speaker_a', 'utterance_a', 'speaker_b', 'utterance_b'],
        num_rows: 13197
    })
    test: Dataset({
        features: ['speaker_a', 'utterance_a', 'speaker_b', 'utterance_b'],
        num_rows: 14504
    })
})


In [10]:
# ============================================================
# Keep ONLY Adult -> Child examples
# ============================================================

def is_adult_to_child(example):
    speaker_a = str(example["speaker_a"]).strip().lower()
    speaker_b = str(example["speaker_b"]).strip().lower()

    return (
        speaker_a == "adult"
        and speaker_b == "child"
    )



dataset = dataset.filter(
    is_adult_to_child
)

print("\nAfter adult -> child filtering:")
print(dataset)




After adult -> child filtering:
DatasetDict({
    train: Dataset({
        features: ['speaker_a', 'utterance_a', 'speaker_b', 'utterance_b'],
        num_rows: 122113
    })
    validation: Dataset({
        features: ['speaker_a', 'utterance_a', 'speaker_b', 'utterance_b'],
        num_rows: 6494
    })
    test: Dataset({
        features: ['speaker_a', 'utterance_a', 'speaker_b', 'utterance_b'],
        num_rows: 7139
    })
})


In [11]:
# ============================================================
# Shuffle and create fixed train/validation/test sets
# ============================================================

all_data = dataset["train"].shuffle(
    seed=SEED
)

required_size = (
    TRAIN_SIZE
    + VAL_SIZE
    + TEST_SIZE
)

if len(all_data) < required_size:
    raise ValueError(
        f"Not enough adult->child examples. "
        f"Need {required_size}, "
        f"but only {len(all_data)} are available."
    )


train_dataset = all_data.select(
    range(0, TRAIN_SIZE)
)

val_dataset = all_data.select(
    range(
        TRAIN_SIZE,
        TRAIN_SIZE + VAL_SIZE,
    )
)

test_dataset = all_data.select(
    range(
        TRAIN_SIZE + VAL_SIZE,
        TRAIN_SIZE + VAL_SIZE + TEST_SIZE,
    )
)


print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))


Dataset sizes:
Train: 10000
Validation: 1000
Test: 1000


In [12]:
# ============================================================
# Response-only SFT tokenization
# ============================================================

def tokenize_example(example):
    adult_text = example["utterance_a"]
    child_text = example["utterance_b"]

    # Prompt
    prompt = (
        f"Adult: {adult_text}\n"
        f"Child:"
    )

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
    )["input_ids"]

    # Response.
    #
    # Leading space preserves GPT-2's normal BPE behavior
    # for a word appearing after "Child:".
    response_ids = tokenizer(
        " " + child_text,
        add_special_tokens=False,
    )["input_ids"]

    # Explicit EOS target.
    response_ids = (
        response_ids
        + [tokenizer.eos_token_id]
    )

    # Complete causal-LM sequence.
    input_ids = (
        prompt_ids
        + response_ids
    )

    # Response-only loss.
    #
    # Prompt positions are ignored by CrossEntropyLoss.
    labels = (
        [-100] * len(prompt_ids)
        + response_ids
    )

    # Truncate input and labels together.
    input_ids = input_ids[:MAX_LENGTH]
    labels = labels[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "labels": labels,
    }


train_dataset = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)

val_dataset = val_dataset.map(
    tokenize_example,
    remove_columns=val_dataset.column_names,
)

test_dataset = test_dataset.map(
    tokenize_example,
    remove_columns=test_dataset.column_names,
)



In [13]:
# ============================================================
# Dynamic padding collator
# ============================================================

def collate_fn(features):
    input_ids = [
        item["input_ids"]
        for item in features
    ]

    labels = [
        item["labels"]
        for item in features
    ]

    # Pad input_ids with GPT-2's pad token.
    batch_input_ids = tokenizer.pad(
        {
            "input_ids": input_ids,
        },
        padding=True,
        return_tensors="pt",
    )["input_ids"]

    max_length = batch_input_ids.shape[1]

    # Pad labels with -100 so padding does not contribute
    # to the loss.
    padded_labels = []

    for label in labels:
        padded = (
            label
            + [-100] * (max_length - len(label))
        )

        padded_labels.append(padded)

    batch_labels = torch.tensor(
        padded_labels,
        dtype=torch.long,
    )

    attention_mask = (
        batch_input_ids
        != tokenizer.pad_token_id
    ).long()

    return {
        "input_ids": batch_input_ids,
        "labels": batch_labels,
        "attention_mask": attention_mask,
    }




In [15]:
import bitsandbytes
print(bitsandbytes.__version__)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

0.50.2
Using device: cuda


In [16]:

# ============================================================
# Load GPT-2 in 4-bit
# ============================================================

print("\nLoading GPT-2 in 4-bit NF4...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.pad_token_id = (
    tokenizer.pad_token_id
)




Loading GPT-2 in 4-bit NF4...


model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [17]:
# ============================================================
# Prepare quantized model for k-bit training
# ============================================================

model = prepare_model_for_kbit_training(
    model
)


In [18]:
# ============================================================
# D1 LoRA target modules
#
# MLP only, all 12 GPT-2 blocks.
#
# Block:
#
#   Attention
#       c_attn
#       c_proj
#
#   MLP
#       c_fc    <-- LoRA
#       c_proj  <-- LoRA
# ============================================================

target_modules = []

for block_idx in range(12):
    target_modules.append(
        f"transformer.h.{block_idx}.mlp.c_fc"
    )

    target_modules.append(
        f"transformer.h.{block_idx}.mlp.c_proj"
    )


print("\nLoRA target modules:")

for module in target_modules:
    print(" ", module)




LoRA target modules:
  transformer.h.0.mlp.c_fc
  transformer.h.0.mlp.c_proj
  transformer.h.1.mlp.c_fc
  transformer.h.1.mlp.c_proj
  transformer.h.2.mlp.c_fc
  transformer.h.2.mlp.c_proj
  transformer.h.3.mlp.c_fc
  transformer.h.3.mlp.c_proj
  transformer.h.4.mlp.c_fc
  transformer.h.4.mlp.c_proj
  transformer.h.5.mlp.c_fc
  transformer.h.5.mlp.c_proj
  transformer.h.6.mlp.c_fc
  transformer.h.6.mlp.c_proj
  transformer.h.7.mlp.c_fc
  transformer.h.7.mlp.c_proj
  transformer.h.8.mlp.c_fc
  transformer.h.8.mlp.c_proj
  transformer.h.9.mlp.c_fc
  transformer.h.9.mlp.c_proj
  transformer.h.10.mlp.c_fc
  transformer.h.10.mlp.c_proj
  transformer.h.11.mlp.c_fc
  transformer.h.11.mlp.c_proj


In [19]:
# ============================================================
# LoRA configuration
# ============================================================

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=target_modules,
)



In [20]:
# ============================================================
# Attach LoRA adapters
# ============================================================

model = get_peft_model(
    model,
    lora_config,
)


In [21]:
# ============================================================
# Trainable parameter inspection
# ============================================================

model.print_trainable_parameters()



trainable params: 737,280 || all params: 125,177,088 || trainable%: 0.5890


In [22]:
# ============================================================
# Sanity checks
# ============================================================

trainable_names = [
    name
    for name, param in model.named_parameters()
    if param.requires_grad
]


# Only LoRA parameters should be trainable.
assert all(
    "lora_" in name
    for name in trainable_names
)


# D1 should only modify MLP layers.
assert all(
    ".mlp." in name
    for name in trainable_names
)


# Both MLP projections should have LoRA.
assert any(
    ".mlp.c_fc." in name
    for name in trainable_names
)

assert any(
    ".mlp.c_proj." in name
    for name in trainable_names
)



In [24]:
# ============================================================
# Training arguments
# ============================================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=100,

    report_to="none",


    # Keep this explicit.
    #
    # QLoRA's quantized weights are handled by bitsandbytes.
    # We do not additionally enable fp16/bf16 here.
    fp16=False,
    bf16=False,

    seed=SEED,
)



In [26]:
# ============================================================
# Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    processing_class=tokenizer,
    data_collator=collate_fn,
)




In [27]:
# ============================================================
# Train
# ============================================================

trainer.train()



[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.046747,2.868170


TrainOutput(global_step=2500, training_loss=3.1139368774414065, metrics={'train_runtime': 253.4558, 'train_samples_per_second': 39.455, 'train_steps_per_second': 9.864, 'total_flos': 133417472901120.0, 'train_loss': 3.1139368774414065, 'epoch': 1.0})

In [28]:
# ============================================================
# Validation evaluation
# ============================================================

print("\nValidation evaluation:")

val_metrics = trainer.evaluate()

print(val_metrics)





Validation evaluation:


Training Loss,Validation Loss,Epoch
3.046747,2.868170,1


{'eval_loss': 2.8681700229644775}


In [29]:
# ============================================================
# Held-out test evaluation
# ============================================================

print("\nTest evaluation:")

test_metrics = trainer.evaluate(
    eval_dataset=test_dataset
)

print(test_metrics)





Test evaluation:


Training Loss,Validation Loss,Epoch
3.046747,2.964664,1


{'eval_loss': 2.964663505554199}


In [30]:
# ============================================================
# Save adapter
# ============================================================

trainer.save_model(
    OUTPUT_DIR
)

tokenizer.save_pretrained(
    OUTPUT_DIR
)


('./gpt2-childes-qlora-d1/tokenizer_config.json',
 './gpt2-childes-qlora-d1/tokenizer.json')

In [31]:
# ============================================================
# Generation helper
# ============================================================

def generate_child_response(adult_text):
    prompt = (
        f"Adult: {adult_text}\n"
        f"Child:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    )

    # device_map="auto" may place the model somewhere other
    # than CPU, so put inputs on the same device as the model's
    # input embedding layer.
    input_device = (
        model.get_input_embeddings()
        .weight
        .device
    )


    inputs = {
        key: value.to(input_device)
        for key, value in inputs.items()
    }

    prompt_length = (
        inputs["input_ids"].shape[1]
    )

    generated_ids = model.generate(
        **inputs,

        max_new_tokens=50,

        do_sample=True,
        temperature=0.8,
        top_p=0.9,

        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )


    # Keep only newly generated response tokens.
    new_token_ids = generated_ids[
        0,
        prompt_length:
    ]

    response = tokenizer.decode(
        new_token_ids,
        skip_special_tokens=False,
    )

    return response



In [32]:
# ============================================================
# Example generation
# ============================================================

print("\nExample generation:")

print(
    generate_child_response(
        "What are you doing?"
    )
)




Example generation:
 i'm getting some .<|endoftext|>


In [33]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [34]:
!jupyter nbconvert --to html /content/drive/MyDrive/"Colab Notebooks"/sft_childes_gpt2_qlora_4bitNF4base_d1.ipynb --output-dir=/content/drive/MyDrive/

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/sft_childes_gpt2_qlora_4bitNF4base_d1.ipynb to html
[NbConvertApp] Writing 386417 bytes to /content/drive/MyDrive/sft_childes_gpt2_qlora_4bitNF4base_d1.html
